# Full Planner — Best Strategy Simulation Notebook

This notebook is dedicated to simulating, evaluating, and discovering the **best overall strategy** for SpaceInvasion Classic across full-empire progression (economy scaling, energy optimization, research unlocks, fleet composition, raiding, colonization, and action scheduling).

## 0. Imports

All Python standard-library and `spaceinvasion` domain imports used across the simulation notebook.

In [ ]:
import contextlib
import datetime
import json
import math
from pathlib import Path
import shutil
import sqlite3
import statistics
import sys
import tempfile
from typing import Any

for _cand in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (_cand / "spaceinvasion" / "__init__.py").is_file():
        if str(_cand) not in sys.path:
            sys.path.insert(0, str(_cand))
        break

from spaceinvasion import game_data
from spaceinvasion.action_graph import build_action_graph, merge_strategy_config_overrides
from spaceinvasion.config import StrategyConfig
from spaceinvasion.fleet import _parse_coords_distance
from spaceinvasion.logging_utils import redact_sensitive
from spaceinvasion.optimizer import encode_empire_state, run_optimizer, simulate_action_sequence
from spaceinvasion.ranking import (
    attach_action_etas,
    compute_action_eta,
    estimate_account_points,
    estimate_hourly_economy,
)
from spaceinvasion.state import EnergyBalance, GameState, PlanetState, Resources
from spaceinvasion.storage import GameDatabase
from spaceinvasion.strategy import PlannedAction, plan_actions

## 1. Utils

Shared path-resolution (`resolve_repo_root`, `resolve_active_data_root`), formatting (`fmt_duration`, `render_table`), JSON parsing (`_safe_json_dict`), statistical (`compute_mean_and_std`), and flight-physics (`estimate_physics_one_way_seconds`) utilities reused across all notebook sections.

In [ ]:
def resolve_repo_root() -> Path:
    """Locate repository root from any working directory without bare `__file__`."""
    cwd = Path.cwd().resolve()
    for cand in (cwd, *cwd.parents):
        if (cand / "spaceinvasion" / "__init__.py").is_file():
            return cand
    for entry in list(sys.path):
        if not entry:
            continue
        p = Path(entry).resolve()
        if (p / "spaceinvasion" / "__init__.py").is_file():
            return p
    return Path(game_data.__file__).resolve().parents[1]


def resolve_active_data_root(repo_root: Path) -> Path:
    """Prefer isolated `cwd` data/state fixtures when present; otherwise use `repo_root`."""
    cwd = Path.cwd().resolve()
    if (cwd / "state" / "spaceinvasion.db").is_file() or (
        cwd / "data" / "attack_history.json"
    ).is_file():
        return cwd
    return repo_root


def fmt_duration(seconds: float) -> str:
    """Format seconds into readable `Xh Ym` or `Xm Ys` strings."""
    if not math.isfinite(seconds) or seconds <= 0.0:
        return "0s"
    sec_i = int(round(seconds))
    if sec_i < 60:
        return f"{sec_i}s"
    mins, rem_s = divmod(sec_i, 60)
    if mins < 60:
        return f"{mins}m {rem_s:02d}s"
    hrs, rem_m = divmod(mins, 60)
    return f"{hrs}h {rem_m:02d}m"


def render_table(headers: list[str], rows: list[list[Any]]) -> str:
    """Render an aligned ASCII grid."""
    str_rows = [[str(c) for c in row] for row in rows]
    widths = [len(h) for h in headers]
    for row in str_rows:
        for idx, cell in enumerate(row):
            if idx < len(widths):
                widths[idx] = max(widths[idx], len(cell))
    sep = "+-" + "-+-".join("-" * w for w in widths) + "-+"
    head = "| " + " | ".join(h.ljust(widths[i]) for i, h in enumerate(headers)) + " |"
    body = [
        "| " + " | ".join((row[i] if i < len(row) else "").ljust(widths[i]) for i in range(len(widths))) + " |"
        for row in str_rows
    ]
    return "\n".join([sep, head, sep, *body, sep])


def _safe_json_dict(raw: Any) -> dict[str, Any]:
    if isinstance(raw, dict):
        return dict(raw)
    if isinstance(raw, str) and raw.strip():
        try:
            parsed = json.loads(raw)
            return dict(parsed) if isinstance(parsed, dict) else {}
        except ValueError:
            return {}
    return {}


def compute_mean_and_std(values: list[float]) -> tuple[float, float, float]:
    """Return `(mean, sample_std, population_std)` safely for any list length."""
    clean = [float(v) for v in values if math.isfinite(float(v))]
    if not clean:
        return 0.0, 0.0, 0.0
    mean_val = float(statistics.fmean(clean))
    sample_std = float(statistics.stdev(clean)) if len(clean) >= 2 else 0.0
    pop_std = float(statistics.pstdev(clean)) if len(clean) >= 1 else 0.0
    return mean_val, sample_std, pop_std


def estimate_physics_one_way_seconds(
    origin_coords: str,
    target_coords: str,
    *,
    min_ship_speed: float = 12_000.0,
    fleet_speed_factor: float = 5.0,
) -> float:
    """SpaceInvasion Classic one-way flight time formula in seconds."""
    dist = max(1.0, float(_parse_coords_distance(origin_coords, target_coords)))
    spd = max(100.0, float(min_ship_speed))
    fac = max(100.0 if fleet_speed_factor > 100.0 else max(1.0, float(fleet_speed_factor)), 1.0)
    return float(round(10.0 + (3500.0 / fac) * math.sqrt((10.0 * dist) / spd)))


REPO_ROOT = resolve_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
ACTIVE_DATA_ROOT = resolve_active_data_root(REPO_ROOT)
LIVE_DB_PATH = ACTIVE_DATA_ROOT / "state" / "spaceinvasion.db"
ATTACK_HISTORY_PATH = ACTIVE_DATA_ROOT / "data" / "attack_history.json"
RESOURCE_KEYS: tuple[str, ...] = ("pig_iron", "metal", "kryptonite", "spice")

print(f"Utils initialized (REPO_ROOT={REPO_ROOT}, ACTIVE_DATA_ROOT={ACTIVE_DATA_ROOT})")

## 2. MER-Aligned Read-Only SQLite Telemetry Queries

Following [documentation/database-mer.md](../documentation/database-mer.md), this cell copies `state/spaceinvasion.db` (+ `-wal` / `-shm` sidecars) into a temporary directory and executes **four relational SQL queries** leveraging the exact keys, soft-flags, and secondary indexes of the 20-table schema:

1. **Colony & Fleet-Slot Capacity (`PLANET_SNAPSHOTS` via `idx_planet_snapshots_pid_ts`)**: Extracts latest colony coordinates, `Fleet base` (`b10`) level (`1 + b10` concurrent flight slots), and engine research levels (`r4`, `r5`, `r6`).
2. **Resolved Combat & Target Classification Join (`BATTLE_REPORTS` $\bowtie$ `GAME_MESSAGES` on `msg_uid` $\bowtie$ `SPY_TARGETS` on `coords`)**: Joins 4-resource plunder, combat fleet losses (`attacker_lost_spice`), and spawned orbital rubble (`debris_spice_eq`) with target status (`is_spacia`, `is_inactive`, `status_color`, `bash_hits`), filtering out soft-merged/own rows (`is_own_planet = 0 AND merged_into = ''`).
3. **Dispatch-to-Origin Colony Join (`RAID_HISTORY` via `idx_raid_history_coords_ts` $\bowtie$ `ACTION_HISTORY` via `idx_action_history_ts`)**: Resolves the exact launching colony `origin_coords` for each dispatched raid rather than assuming a single static origin.
4. **Orbital Rubble & Bash Cooldown Context (`RECYCLE_REPORTS` via `idx_recycle_reports_coords_ts` & `BASH_BLOCKS` via `idx_bash_blocks_coords_ts`)**: Queries supplementary recycler harvests and active server bash blocks.

In [ ]:
SQL_LATEST_COLONY_SNAPSHOT = """
SELECT coords, buildings_json, researches_json
FROM planet_snapshots
ORDER BY ts DESC, id DESC
LIMIT 1
"""

SQL_BATTLE_REPORTS_MER_JOIN = """
SELECT
    br.id,
    br.ts,
    br.msg_uid,
    br.battle_id,
    br.msg_date_text,
    br.target_coords,
    br.outcome,
    br.rounds,
    br.pig_iron,
    br.metal,
    br.kryptonite,
    br.spice,
    br.loot_spice_eq,
    br.attacker_lost_spice,
    br.defender_lost_spice,
    br.debris_spice_eq,
    gm.ingested_ts AS msg_ingested_ts,
    gm.deleted_in_game,
    COALESCE(st.status_color, 'unknown') AS target_status_color,
    COALESCE(st.is_inactive, 0) AS target_is_inactive,
    COALESCE(st.is_spacia, 0) AS target_is_spacia,
    COALESCE(st.bash_hits, 0) AS target_bash_hits
FROM battle_reports br
LEFT JOIN game_messages gm ON gm.msg_uid = br.msg_uid
LEFT JOIN spy_targets st
       ON st.coords = br.target_coords
      AND st.is_own_planet = 0
      AND st.merged_into = ''
ORDER BY br.msg_date_text ASC, br.id ASC
"""

SQL_RAID_DISPATCH_ORIGIN_JOIN = """
SELECT
    rh.id,
    rh.target_coords,
    rh.attacked_ts,
    rh.is_exempt,
    rh.outcome,
    rh.loot_json,
    COALESCE(
        (
            SELECT ah.planet_coords
            FROM action_history ah
            WHERE ah.kind = 'ATTACK'
              AND ah.executed = 1
              AND ah.dry_run = 0
              AND ah.planet_coords IS NOT NULL
              AND ah.planet_coords != ''
              AND ABS(ah.ts - rh.attacked_ts) <= 5.0
            ORDER BY ABS(ah.ts - rh.attacked_ts) ASC
            LIMIT 1
        ),
        ?
    ) AS origin_coords
FROM raid_history rh
ORDER BY rh.attacked_ts ASC, rh.id ASC
"""

SQL_RECYCLE_HARVESTS = """
SELECT
    coords,
    recycler_count,
    harvested_pig_iron,
    harvested_metal,
    harvested_kryptonite,
    harvested_spice,
    harvested_spice_eq,
    ts
FROM recycle_reports
ORDER BY ts ASC, id ASC
"""

SQL_ACTIVE_BASH_BLOCKS = """
SELECT COUNT(DISTINCT target_coords)
FROM bash_blocks
WHERE blocked_until_ts > ?
"""


def load_attack_telemetry_readonly(
    db_path: Path,
    json_path: Path,
) -> dict[str, Any]:
    """Execute static MER SQL queries on an isolated temp copy of `spaceinvasion.db`."""
    origin_coords = "02:380:12"
    fleet_base_level = 0
    engine_tech_levels: dict[str, int] = {"r4": 0, "r5": 0, "r6": 0}
    battle_rows: list[dict[str, Any]] = []
    raid_rows: list[dict[str, Any]] = []
    recycle_rows: list[dict[str, Any]] = []
    active_bash_blocks: int = 0
    source_label = "cold_start"

    if db_path.is_file() and db_path.stat().st_size >= 100:
        with tempfile.TemporaryDirectory(prefix="si_fp_ro_") as tmp_dir:
            tmp_root = Path(tmp_dir)
            tmp_db = tmp_root / db_path.name
            shutil.copy2(db_path, tmp_db)
            tmp_db.chmod(0o600)
            for suffix in ("-wal", "-shm", "-journal"):
                sidecar = Path(f"{db_path}{suffix}")
                if sidecar.is_file():
                    tmp_sidecar = Path(f"{tmp_db}{suffix}")
                    shutil.copy2(sidecar, tmp_sidecar)
                    tmp_sidecar.chmod(0o600)

            ro_db = GameDatabase(tmp_db, attack_history_path=tmp_root / "ro_mirror.json")
            try:
                conn = ro_db._conn
                ps = conn.execute(SQL_LATEST_COLONY_SNAPSHOT).fetchone()
                if ps:
                    if ps["coords"]:
                        origin_coords = str(ps["coords"])
                    bld_d = _safe_json_dict(ps["buildings_json"])
                    res_d = _safe_json_dict(ps["researches_json"])
                    fleet_base_level = max(0, int(bld_d.get("b10") or 0))
                    for rk in ("r4", "r5", "r6"):
                        engine_tech_levels[rk] = max(0, int(res_d.get(rk) or 0))

                battle_rows = [dict(r) for r in conn.execute(SQL_BATTLE_REPORTS_MER_JOIN).fetchall()]
                raid_rows = [
                    dict(r)
                    for r in conn.execute(SQL_RAID_DISPATCH_ORIGIN_JOIN, (origin_coords,)).fetchall()
                ]
                recycle_rows = [dict(r) for r in conn.execute(SQL_RECYCLE_HARVESTS).fetchall()]

                ref_ts = max(
                    [0.0]
                    + [float(r.get("attacked_ts") or 0.0) for r in raid_rows]
                    + [float(b.get("ts") or 0.0) for b in battle_rows]
                )
                active_bash_blocks = int(
                    conn.execute(SQL_ACTIVE_BASH_BLOCKS, (ref_ts,)).fetchone()[0]
                )
            finally:
                ro_db.close()

            if battle_rows or raid_rows:
                source_label = "sqlite:mer_joined(battle_reports+raid_history+spy_targets)"

    if not battle_rows and not raid_rows and json_path.is_file():
        raw_obj = _safe_json_dict(json_path.read_text(encoding="utf-8"))
        for row in raw_obj.get("history") or []:
            if isinstance(row, dict):
                raid_rows.append({**dict(row), "origin_coords": origin_coords})
        if raid_rows:
            source_label = "json:attack_history"

    dispatch_timestamps = sorted(
        float(r["attacked_ts"])
        for r in raid_rows
        if isinstance(r.get("attacked_ts"), (int, float)) and math.isfinite(float(r["attacked_ts"]))
    )

    tz_offsets_sec: list[float] = []
    parsed_battle_naive: list[float | None] = []
    for brow in battle_rows:
        dt_txt = str(brow.get("msg_date_text") or "").strip()
        try:
            naive_ts = (
                datetime.datetime.strptime(dt_txt, "%Y-%m-%d %H:%M:%S")
                .replace(tzinfo=datetime.timezone.utc)
                .timestamp()
            )
        except ValueError:
            naive_ts = None
        parsed_battle_naive.append(naive_ts)
        if naive_ts is not None:
            for rrow in raid_rows:
                if str(rrow.get("target_coords") or "") != str(brow.get("target_coords") or ""):
                    continue
                rts = float(rrow.get("attacked_ts") or 0.0)
                cand_h = round((rts - naive_ts) / 3600.0)
                if abs(cand_h) <= 14 and 30.0 <= (naive_ts + cand_h * 3600.0 - rts) <= 3600.0:
                    tz_offsets_sec.append(cand_h * 3600.0)

    server_tz_offset_sec = statistics.median(tz_offsets_sec) if tz_offsets_sec else 0.0

    records: list[dict[str, Any]] = []
    used_raid_ids: set[int] = set()
    if battle_rows:
        matched_pairs_sec_per_sqrtdist: list[float] = []
        provisional_rows: list[tuple[dict[str, Any], float, float | None, str]] = []
        for brow, naive_ts in zip(battle_rows, parsed_battle_naive):
            t_coords = str(brow.get("target_coords") or "")
            impact_ts = (
                (naive_ts + server_tz_offset_sec)
                if naive_ts is not None
                else float(brow.get("ts") or 0.0)
            )
            matched_one_way: float | None = None
            matched_origin = origin_coords
            best_rid: int | None = None
            for idx_r, rrow in enumerate(raid_rows):
                if idx_r in used_raid_ids or str(rrow.get("target_coords") or "") != t_coords:
                    continue
                rts = float(rrow.get("attacked_ts") or 0.0)
                dt_flight = impact_ts - rts
                if 30.0 <= dt_flight <= 7200.0:
                    if matched_one_way is None or dt_flight < matched_one_way:
                        matched_one_way = dt_flight
                        matched_origin = str(rrow.get("origin_coords") or origin_coords)
                        best_rid = idx_r
            if best_rid is not None and matched_one_way is not None:
                used_raid_ids.add(best_rid)
                dist = max(1.0, float(_parse_coords_distance(matched_origin, t_coords)))
                matched_pairs_sec_per_sqrtdist.append(max(1.0, matched_one_way - 10.0) / math.sqrt(dist))
            provisional_rows.append((brow, impact_ts, matched_one_way, matched_origin))

        emp_scale = (
            statistics.median(matched_pairs_sec_per_sqrtdist)
            if matched_pairs_sec_per_sqrtdist
            else None
        )
        for brow, impact_ts, matched_one_way, row_origin in provisional_rows:
            t_coords = str(brow.get("target_coords") or "")
            if matched_one_way is not None:
                one_way_s = matched_one_way
                t_src = "observed_dispatch_pair"
            elif emp_scale is not None:
                dist = max(1.0, float(_parse_coords_distance(row_origin, t_coords)))
                one_way_s = 10.0 + emp_scale * math.sqrt(dist)
                t_src = "calibrated_distance"
            else:
                one_way_s = estimate_physics_one_way_seconds(row_origin, t_coords)
                t_src = "physics_formula"

            pig = float(brow.get("pig_iron") or 0.0)
            met = float(brow.get("metal") or 0.0)
            kry = float(brow.get("kryptonite") or 0.0)
            spi = float(brow.get("spice") or 0.0)
            sp_eq = float(
                brow.get("loot_spice_eq")
                if brow.get("loot_spice_eq") is not None
                else game_data.spice_equivalent({"pig_iron": pig, "metal": met, "kryptonite": kry, "spice": spi})
            )
            lost_sp = float(brow.get("attacker_lost_spice") or 0.0)
            debris_sp = float(brow.get("debris_spice_eq") or 0.0)
            outcome_str = str(brow.get("outcome") or "").strip().lower()
            won = outcome_str in ("win", "victory")
            is_spacia = bool(brow.get("target_is_spacia"))
            is_inactive = bool(brow.get("target_is_inactive")) or str(brow.get("target_status_color") or "") == "yellow"
            target_class = "spacia" if is_spacia else ("yellow_inactive" if is_inactive else "active_colony")
            records.append(
                {
                    "event_ts": impact_ts,
                    "origin_coords": row_origin,
                    "target_coords": t_coords,
                    "target_class": target_class,
                    "won": won,
                    "outcome": outcome_str or ("win" if won else "loss"),
                    "pig_iron": pig,
                    "metal": met,
                    "kryptonite": kry,
                    "spice": spi,
                    "spice_eq": sp_eq,
                    "attacker_lost_spice": lost_sp,
                    "net_spice_eq": sp_eq - lost_sp,
                    "debris_spice_eq": debris_sp,
                    "one_way_sec": float(one_way_s),
                    "round_trip_sec": float(2.0 * one_way_s),
                    "time_source": t_src,
                }
            )
    else:
        for rrow in raid_rows:
            t_coords = str(rrow.get("target_coords") or "")
            row_origin = str(rrow.get("origin_coords") or origin_coords)
            loot_d = _safe_json_dict(rrow.get("loot") if "loot" in rrow else rrow.get("loot_json"))
            pig = float(loot_d.get("pig_iron") or 0.0)
            met = float(loot_d.get("metal") or 0.0)
            kry = float(loot_d.get("kryptonite") or 0.0)
            spi = float(loot_d.get("spice") or 0.0)
            sp_eq = float(
                game_data.spice_equivalent(
                    {"pig_iron": pig, "metal": met, "kryptonite": kry, "spice": spi}
                )
            )
            outcome_txt = redact_sensitive(str(rrow.get("outcome") or "")).lower()
            won = bool(rrow.get("won")) if "won" in rrow else ("defeat" not in outcome_txt and "loss" not in outcome_txt)
            one_way_s = estimate_physics_one_way_seconds(row_origin, t_coords)
            records.append(
                {
                    "event_ts": float(rrow.get("attacked_ts") or 0.0),
                    "origin_coords": row_origin,
                    "target_coords": t_coords,
                    "target_class": "exempt_farm" if bool(rrow.get("is_exempt")) else "active_colony",
                    "won": won,
                    "outcome": "win" if won else "loss",
                    "pig_iron": pig,
                    "metal": met,
                    "kryptonite": kry,
                    "spice": spi,
                    "spice_eq": sp_eq,
                    "attacker_lost_spice": 0.0,
                    "net_spice_eq": sp_eq,
                    "debris_spice_eq": 0.0,
                    "one_way_sec": float(one_way_s),
                    "round_trip_sec": float(2.0 * one_way_s),
                    "time_source": "physics_formula",
                }
            )

    records.sort(key=lambda r: float(r["event_ts"]))
    if not dispatch_timestamps:
        dispatch_timestamps = [float(r["event_ts"]) for r in records if float(r["event_ts"]) > 0.0]
    return {
        "records": records,
        "dispatch_timestamps": dispatch_timestamps,
        "origin_coords": origin_coords,
        "fleet_base_level": fleet_base_level,
        "fleet_slot_capacity": 1 + fleet_base_level,
        "engine_tech_levels": engine_tech_levels,
        "recycle_rows": recycle_rows,
        "active_bash_blocks": active_bash_blocks,
        "source_label": source_label,
    }


telemetry_bundle = load_attack_telemetry_readonly(LIVE_DB_PATH, ATTACK_HISTORY_PATH)
attack_records: list[dict[str, Any]] = telemetry_bundle["records"]
attack_dispatch_timestamps: list[float] = telemetry_bundle["dispatch_timestamps"]
origin_planet_coords: str = telemetry_bundle["origin_coords"]
fleet_slot_capacity: int = int(telemetry_bundle["fleet_slot_capacity"])
recycle_records: list[dict[str, Any]] = telemetry_bundle["recycle_rows"]
telemetry_source: str = telemetry_bundle["source_label"]

print(
    render_table(
        ["MER Telemetry Property", "Value"],
        [
            ["Telemetry Source", telemetry_source],
            ["Primary Origin Colony", origin_planet_coords],
            ["Fleet Base Level / Flight Slots", f"b10 L{telemetry_bundle['fleet_base_level']} -> {fleet_slot_capacity} concurrent slots"],
            ["Engine Tech Levels (r4 / r5 / r6)", f"{telemetry_bundle['engine_tech_levels']['r4']} / {telemetry_bundle['engine_tech_levels']['r5']} / {telemetry_bundle['engine_tech_levels']['r6']}"],
            ["Joined Combat Attack Records", len(attack_records)],
            ["Distinct Targets / Active Bash Blocks", f"{len({r['target_coords'] for r in attack_records})} targets / {telemetry_bundle['active_bash_blocks']} blocked"],
            ["Archived Recycle Harvest Reports", len(recycle_records)],
        ],
    )
)

## 3. Mean & Standard Deviation of Resources Achieved in Attacks (Gross, Net & Rubble)

Using the `BATTLE_REPORTS` schema from [documentation/database-mer.md](../documentation/database-mer.md), this section computes the **sample mean ($\mu_r$)** and **standard deviation ($s_r$)** across:

- **Gross Plunder**: `pig_iron`, `metal`, `kryptonite`, `spice`, and `spice_eq` (`6 : 2.5 : 3 : 1` Merchant parity).
- **Net Combat Yield (`net_spice_eq`)**: $\text{loot\_spice\_eq} - \text{attacker\_lost\_spice}$, accounting for fleet losses suffered in unsuccessful or costly engagements.
- **Orbital Rubble (`debris_spice_eq` + `RECYCLE_REPORTS`)**: Combat-spawned debris fields and verified `Recycler` (`s11`) harvests (`harvested_spice_eq`).

In [ ]:
METRIC_KEYS: tuple[str, ...] = (*RESOURCE_KEYS, "spice_eq", "net_spice_eq", "debris_spice_eq")
winning_attack_records: list[dict[str, Any]] = [r for r in attack_records if bool(r.get("won"))]

attack_resource_mean: dict[str, float] = {}
attack_resource_std: dict[str, float] = {}
attack_resource_pop_std: dict[str, float] = {}
winning_attack_resource_mean: dict[str, float] = {}
winning_attack_resource_std: dict[str, float] = {}

resource_stat_rows: list[list[Any]] = []
for key in METRIC_KEYS:
    all_vals = [float(r.get(key) or 0.0) for r in attack_records]
    win_vals = [float(r.get(key) or 0.0) for r in winning_attack_records]

    mu_all, sd_all, psd_all = compute_mean_and_std(all_vals)
    mu_win, sd_win, _ = compute_mean_and_std(win_vals)

    attack_resource_mean[key] = mu_all
    attack_resource_std[key] = sd_all
    attack_resource_pop_std[key] = psd_all
    winning_attack_resource_mean[key] = mu_win
    winning_attack_resource_std[key] = sd_win

    resource_stat_rows.append(
        [
            key,
            f"{mu_all:,.2f}",
            f"{sd_all:,.2f}",
            f"{psd_all:,.2f}",
            f"{mu_win:,.2f}",
            f"{sd_win:,.2f}",
        ]
    )

recycle_spice_eq_mean, recycle_spice_eq_std, _ = compute_mean_and_std(
    [float(r.get("harvested_spice_eq") or 0.0) for r in recycle_records]
)

print("3. Attack Resource Loot & Net Combat Yield (All Attacks vs. Winning Attacks)")
print(
    render_table(
        [
            "Metric / Resource",
            "Mean (All)",
            "Sample Std (All)",
            "Pop Std (All)",
            "Mean (Wins Only)",
            "Sample Std (Wins Only)",
        ],
        resource_stat_rows,
    )
)
if recycle_records:
    print(
        f"Supplementary RECYCLE_REPORTS ({len(recycle_records)} harvests): "
        f"mean={recycle_spice_eq_mean:,.2f} SpiceEq, sample_std={recycle_spice_eq_std:,.2f} SpiceEq"
    )

## 4. Mean & Standard Deviation of Attack Time

To accurately simulate fleet-slot capacity (`1 + Fleet base b10` concurrent flights) and raid throughput per hour, we measure two complementary time distributions:

1. **Fleet Mission Flight Time (`one_way_sec` & `round_trip_sec`)**:
   - `one_way_sec` ($\mu_{t,\text{one-way}}, s_{t,\text{one-way}}$): Elapsed flight time from dispatch at origin colony to combat impact at target coordinates (measured directly from paired `raid_history -> battle_reports` timestamps when available, or SpaceInvasion Classic flight physics otherwise).
   - `round_trip_sec` ($\mu_{t,\text{round-trip}}, s_{t,\text{round-trip}}$): Full round-trip sortie duration ($2 \times t_{\text{one-way}}$), which is the exact duration a fleet slot remains occupied before plunder is unloaded at home.
2. **Inter-Attack Dispatch Cadence (`inter_attack_sec`)**:
   - Mean and standard deviation of real wall-clock time ($\Delta t_k = t_{k} - t_{k-1}$) between consecutive attack dispatches in historical telemetry.

In [ ]:
one_way_times_sec: list[float] = [float(r["one_way_sec"]) for r in attack_records]
round_trip_times_sec: list[float] = [float(r["round_trip_sec"]) for r in attack_records]
inter_attack_intervals_sec: list[float] = [
    attack_dispatch_timestamps[idx] - attack_dispatch_timestamps[idx - 1]
    for idx in range(1, len(attack_dispatch_timestamps))
    if attack_dispatch_timestamps[idx] > attack_dispatch_timestamps[idx - 1]
]

attack_one_way_time_mean_sec, attack_one_way_time_std_sec, attack_one_way_time_pop_std_sec = (
    compute_mean_and_std(one_way_times_sec)
)
attack_time_mean_sec, attack_time_std_sec, attack_time_pop_std_sec = (
    compute_mean_and_std(round_trip_times_sec)
)
inter_attack_time_mean_sec, inter_attack_time_std_sec, inter_attack_time_pop_std_sec = (
    compute_mean_and_std(inter_attack_intervals_sec)
)

time_stat_rows: list[list[Any]] = [
    [
        "One-Way Flight Time (Dispatch -> Impact)",
        len(one_way_times_sec),
        f"{attack_one_way_time_mean_sec:,.1f}s ({fmt_duration(attack_one_way_time_mean_sec)})",
        f"{attack_one_way_time_std_sec:,.1f}s ({fmt_duration(attack_one_way_time_std_sec)})",
        f"{attack_one_way_time_pop_std_sec:,.1f}s",
    ],
    [
        "Round-Trip Attack Time (Slot Occupation)",
        len(round_trip_times_sec),
        f"{attack_time_mean_sec:,.1f}s ({fmt_duration(attack_time_mean_sec)})",
        f"{attack_time_std_sec:,.1f}s ({fmt_duration(attack_time_std_sec)})",
        f"{attack_time_pop_std_sec:,.1f}s",
    ],
    [
        "Inter-Attack Dispatch Interval",
        len(inter_attack_intervals_sec),
        f"{inter_attack_time_mean_sec:,.1f}s ({fmt_duration(inter_attack_time_mean_sec)})",
        f"{inter_attack_time_std_sec:,.1f}s ({fmt_duration(inter_attack_time_std_sec)})",
        f"{inter_attack_time_pop_std_sec:,.1f}s",
    ],
]

print("4. Attack Time & Cadence Statistics")
print(
    render_table(
        ["Time Metric", "Samples", "Mean", "Sample Std", "Pop Std"],
        time_stat_rows,
    )
)

## 5. Attack Success Rate

This section evaluates the empirical **Attack Success Rate** across all resolved combat engagements:

$$\hat{p}_{\text{win}} = \frac{W}{N}, \qquad \text{SE}(\hat{p}_{\text{win}}) = \sqrt{\frac{\hat{p}_{\text{win}}(1 - \hat{p}_{\text{win}})}{N}}, \qquad \tilde{p}_{\text{win}} = \frac{W + 1}{N + 2}$$

- **`attack_success_rate` ($\hat{p}_{\text{win}}$)**: Observed fraction of attacks ending in combat victory (`won == True`).
- **`attack_success_rate_std_err`**: Binomial standard error quantifying sample uncertainty around $\hat{p}_{\text{win}}$.
- **`attack_success_rate_smoothed` ($\tilde{p}_{\text{win}}$)**: Laplace-smoothed (`Beta(1, 1)`) estimate preventing degenerate `0%` or `100%` priors when `N` is small.
- **`ATTACK_SIM_PARAMS`**: Bundles the resource means/stds, time means/stds, success rates, and expected `SpiceEq` yield per fleet-slot hour into a single dictionary ready for downstream full-planner simulations.

In [ ]:
attack_attempts: int = len(attack_records)
attack_wins: int = sum(1 for r in attack_records if bool(r.get("won")))
attack_losses: int = attack_attempts - attack_wins

attack_success_rate: float = (
    float(attack_wins) / float(attack_attempts) if attack_attempts > 0 else 0.0
)
attack_success_rate_std_err: float = (
    math.sqrt(attack_success_rate * (1.0 - attack_success_rate) / float(attack_attempts))
    if attack_attempts > 0
    else 0.0
)
attack_success_rate_smoothed: float = float(attack_wins + 1) / float(attack_attempts + 2)

per_target_success_rate: dict[str, dict[str, Any]] = {}
per_class_success_rate: dict[str, dict[str, Any]] = {}
for rec in attack_records:
    coords = str(rec.get("target_coords") or "unknown")
    t_cls = str(rec.get("target_class") or "unknown")
    for group_map, group_key in ((per_target_success_rate, coords), (per_class_success_rate, t_cls)):
        bucket = group_map.setdefault(
            group_key,
            {"attempts": 0, "wins": 0, "losses": 0, "loot_spice_eq": [], "net_spice_eq": []},
        )
        bucket["attempts"] += 1
        if bool(rec.get("won")):
            bucket["wins"] += 1
        else:
            bucket["losses"] += 1
        bucket["loot_spice_eq"].append(float(rec.get("spice_eq") or 0.0))
        bucket["net_spice_eq"].append(float(rec.get("net_spice_eq") or 0.0))

target_sr_rows: list[list[Any]] = []
for coords in sorted(per_target_success_rate.keys()):
    b = per_target_success_rate[coords]
    att = int(b["attempts"])
    wins = int(b["wins"])
    sr = float(wins) / float(att) if att > 0 else 0.0
    mu_sp, sd_sp, _ = compute_mean_and_std(list(b["loot_spice_eq"]))
    mu_net, _, _ = compute_mean_and_std(list(b["net_spice_eq"]))
    b["success_rate"] = sr
    b["mean_spice_eq"] = mu_sp
    b["std_spice_eq"] = sd_sp
    b["mean_net_spice_eq"] = mu_net
    target_sr_rows.append(
        [
            coords,
            f"{wins} / {att}",
            f"{sr:.2%}",
            f"{mu_sp:,.2f}",
            f"{sd_sp:,.2f}",
            f"{mu_net:+,.2f}",
        ]
    )

class_sr_rows: list[list[Any]] = []
for t_cls in sorted(per_class_success_rate.keys()):
    cb = per_class_success_rate[t_cls]
    att = int(cb["attempts"])
    wins = int(cb["wins"])
    sr = float(wins) / float(att) if att > 0 else 0.0
    mu_sp, _, _ = compute_mean_and_std(list(cb["loot_spice_eq"]))
    mu_net, _, _ = compute_mean_and_std(list(cb["net_spice_eq"]))
    cb["success_rate"] = sr
    cb["mean_spice_eq"] = mu_sp
    cb["mean_net_spice_eq"] = mu_net
    class_sr_rows.append([t_cls, f"{wins} / {att}", f"{sr:.2%}", f"{mu_sp:,.2f}", f"{mu_net:+,.2f}"])

expected_raids_per_slot_hr: float = (
    3600.0 / attack_time_mean_sec if attack_time_mean_sec > 0.0 else 0.0
)
expected_spice_eq_per_slot_hr: float = (
    expected_raids_per_slot_hr * attack_resource_mean["spice_eq"]
)
expected_net_spice_eq_per_slot_hr: float = (
    expected_raids_per_slot_hr * attack_resource_mean["net_spice_eq"]
)
max_empire_gross_spice_eq_hr: float = expected_spice_eq_per_slot_hr * float(fleet_slot_capacity)

ATTACK_SIM_PARAMS: dict[str, Any] = {
    "attempts": attack_attempts,
    "wins": attack_wins,
    "losses": attack_losses,
    "success_rate": attack_success_rate,
    "success_rate_std_err": attack_success_rate_std_err,
    "success_rate_smoothed": attack_success_rate_smoothed,
    "fleet_slot_capacity": fleet_slot_capacity,
    "resource_mean": dict(attack_resource_mean),
    "resource_std": dict(attack_resource_std),
    "winning_resource_mean": dict(winning_attack_resource_mean),
    "winning_resource_std": dict(winning_attack_resource_std),
    "recycle_spice_eq_mean": recycle_spice_eq_mean,
    "recycle_spice_eq_std": recycle_spice_eq_std,
    "one_way_time_mean_sec": attack_one_way_time_mean_sec,
    "one_way_time_std_sec": attack_one_way_time_std_sec,
    "round_trip_time_mean_sec": attack_time_mean_sec,
    "round_trip_time_std_sec": attack_time_std_sec,
    "inter_attack_time_mean_sec": inter_attack_time_mean_sec,
    "inter_attack_time_std_sec": inter_attack_time_std_sec,
    "expected_raids_per_slot_hr": expected_raids_per_slot_hr,
    "expected_spice_eq_per_slot_hr": expected_spice_eq_per_slot_hr,
    "expected_net_spice_eq_per_slot_hr": expected_net_spice_eq_per_slot_hr,
    "max_empire_gross_spice_eq_hr": max_empire_gross_spice_eq_hr,
}

print("5. Attack Success Rate & Expected Fleet-Slot Yield")
print(
    render_table(
        ["Metric", "Value"],
        [
            ["Total Attacks (Wins / Losses)", f"{attack_attempts} ({attack_wins} wins / {attack_losses} losses)"],
            ["Empirical Attack Success Rate", f"{attack_success_rate:.2%} (+/- {attack_success_rate_std_err:.2%} SE)"],
            ["Laplace-Smoothed Success Rate", f"{attack_success_rate_smoothed:.2%}"],
            ["Max Sorties per Fleet Slot / Hour", f"{expected_raids_per_slot_hr:.2f} raids/hr (mean round-trip {fmt_duration(attack_time_mean_sec)})"],
            ["Gross / Net Loot per Slot-Hour", f"{expected_spice_eq_per_slot_hr:,.1f} gross / {expected_net_spice_eq_per_slot_hr:+,.1f} net SpiceEq/hr"],
            [f"Full Capacity Ceiling ({fleet_slot_capacity} Slots)", f"{max_empire_gross_spice_eq_hr:,.1f} gross SpiceEq/hr"],
        ],
    )
)
if class_sr_rows:
    print("\nSuccess Rate by SPY_TARGETS Class (MER Join)")
    print(
        render_table(
            ["Target Class", "Wins / Attempts", "Success Rate", "Mean Gross SpEq", "Mean Net SpEq"],
            class_sr_rows,
        )
    )
if target_sr_rows:
    print("\nPer-Target Success Rate & Net Combat Yield")
    print(
        render_table(
            ["Target Coords", "Wins / Attempts", "Success Rate", "Mean Gross SpEq", "Sample Std SpEq", "Mean Net SpEq"],
            target_sr_rows,
        )
    )